# TapForge local OCR pipeline benchmark

This notebook compares reproducible, synthetic-card results. It contains no production contacts or credentials.

In [1]:
import json
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
RESULTS = NOTEBOOK_DIR / 'results'
names = [
    'ppocr', 'ppocr-phi-blocks', 'paddle-image',
    'paddle-image-capped', 'phi4-image', 'hunyuan-image',
]
metrics = {name: json.loads((RESULTS / f'{name}.metrics.json').read_text()) for name in names}
runtime = json.loads((NOTEBOOK_DIR / 'runtime-evidence.json').read_text())
len(metrics), runtime['host']

(6, 'thinkstationpgx-ab59')

In [2]:
columns = ('candidate', 'success', 'json', 'field_exact', 'empty_hallucination', 'raw_coverage', 'cer', 'p95_ms')
print(' | '.join(columns))
print(' | '.join(['---'] * len(columns)))
for name in names:
    item = metrics[name]
    values = (
        name, item['success_rate'], item['semantic_json_rate'],
        item['field_exact_accuracy'], item['hallucination_rate_on_empty'],
        item['raw_field_coverage'], item['transcription_cer'], item['latency_p95_ms'],
    )
    print(' | '.join(f'{value:.4f}' if isinstance(value, float) else str(value) for value in values))

candidate | success | json | field_exact | empty_hallucination | raw_coverage | cer | p95_ms
--- | --- | --- | --- | --- | --- | --- | ---
ppocr | 1.0000 | 0.0000 | 0.0000 | 0.0000 | 1.0000 | 0.1087 | 4468.6047
ppocr-phi-blocks | 1.0000 | 1.0000 | 0.7692 | 0.2326 | 0.0000 | 0.0000 | 1992.4199
paddle-image | 1.0000 | 0.0000 | 0.0000 | 0.0000 | 0.8033 | 0.1844 | 26446.1857
paddle-image-capped | 1.0000 | 0.0000 | 0.0000 | 0.0000 | 0.8033 | 0.1844 | 2490.6936
phi4-image | 1.0000 | 1.0000 | 0.1250 | 0.8605 | 0.0000 | 0.0000 | 2434.6048
hunyuan-image | 1.0000 | 0.5000 | 0.4615 | 1.0000 | 1.0000 | 1.8118 | 1368.6825


In [3]:
ppocr = metrics['ppocr']
semantic = metrics['ppocr-phi-blocks']
combined_p95_ms = ppocr['latency_p95_ms'] + semantic['latency_p95_ms']
gates = {
    'ppocr_success_100pct': ppocr['success_rate'] == 1.0,
    'ppocr_raw_field_coverage_100pct': ppocr['raw_field_coverage'] == 1.0,
    'ppocr_peak_below_2gb': ppocr['peak_rss_mb'] < 2048,
    'semantic_json_100pct': semantic['semantic_json_rate'] == 1.0,
    'combined_p95_below_9s': combined_p95_ms < 9000,
    'paddle_image_rejected': metrics['paddle-image']['semantic_json_rate'] == 0.0,
    'phi_image_rejected': metrics['phi4-image']['hallucination_rate_on_empty'] > 0.5,
    'hunyuan_rejected': metrics['hunyuan-image']['semantic_json_rate'] < 1.0,
}
print(f'combined_p95_ms={combined_p95_ms:.3f}')
for gate, passed in gates.items():
    print(f'{gate}={passed}')
assert all(gates.values())

combined_p95_ms=6461.025
ppocr_success_100pct=True
ppocr_raw_field_coverage_100pct=True
ppocr_peak_below_2gb=True
semantic_json_100pct=True
combined_p95_below_9s=True
paddle_image_rejected=True
phi_image_rejected=True
hunyuan_rejected=True


## Decision

Use PP-OCRv6 medium for raw text and bounding boxes, then Phi-4 in text-only mode for semantic organization. Deterministic parsers own exact fields, and unsupported semantic values must require human review. Direct-image PaddleOCR-VL, Phi-4, and HunyuanOCR do not satisfy the production contract.